In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
#sns.set_palette("husl")
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')
import re

In [3]:
df = pd.read_csv('../data/SraRunTable.csv', index_col=False)
total_bytes = df['Bytes'].sum()
print(f"Общий объем данных: {total_bytes/1e12:.4f} TB")

Общий объем данных: 23.9746 TB


### Чистим от 16S rRNA было 2807 - стало 1707

In [ ]:
deleted_rows = df[df['target_gene (exp)'] == '16S rRNA']
df = df[df['target_gene (exp)'] != '16S rRNA']
#df
deleted_rows

### Убираем запуски с аномально большим количеством баз (больше 30 гигабаз и весят по 50-150 гигабайт) (пока не убираем)

In [141]:
#df = df[df['Bases'].apply(lambda x: x<30000000000)]
#df

### Удаляем хозяев мышь и выдру

In [ ]:
df = df[(df['host'] != 'Mus musculus') & (df['host'] !='southern_sea_otter')]
df

### Удаляем проект PRJNA1214429 с 311 архивами, в которых все прочтения бактериальные и PRJNA665586 (4 архива) просто мазок из пещеры - вирусных прочтений нет.

In [ ]:
df = df[(df['BioProject'] != 'PRJNA1214429') & (df['BioProject'] != 'PRJNA665586')]
df

### Удаляем проекты, где организм бактерии

In [14]:
pattern = r'(Escherichia coli|Lactobacillus|Lactococcus sp\.|Escherichia marmotae|Clostridium sp\.|Enterobacteriaceae bacterium|Streptomyces sp\.|Staphylococcus nepalensis)'

mask_org = ~df['Organism'].str.contains(pattern, case=False, na=False, regex=True)

#mask_source = ~df['source_name'].str.contains(r'(lung|blood)', case=False, na=False)

mask_tissue = ~df['tissue'].str.contains(r'(lung|blood)', case=False, na=False)

mask = mask_org & mask_tissue

filtered_df = df[mask]
removed = df[~mask]

print(f"Удалено {len(removed)} строк:")

df = filtered_df

Удалено 0 строк:


In [15]:
df

,Run,Assay Type,AvgSpotLen,Bases,BioSample,Bytes,Center Name,Consent,DATASTORE filetype,DATASTORE provider,...,additional_component_added_to_sample,altitude,BIOME,env_package,feature,Investigation_type,Label,material,sample_comment,spike_in_level_of_influenza_a_virus
0,SRR27491126,AMPLICON,300,3796245900,SAMN39397543,2388715895,INSTITUT PASTEUR OF SHANGHAI,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SRR27491127,AMPLICON,300,2721147000,SAMN39397542,1707201651,INSTITUT PASTEUR OF SHANGHAI,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SRR27491128,AMPLICON,300,4709894100,SAMN39397541,2950874629,INSTITUT PASTEUR OF SHANGHAI,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR27491129,AMPLICON,300,4779211500,SAMN39397540,3015086016,INSTITUT PASTEUR OF SHANGHAI,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR27491130,AMPLICON,300,5456230500,SAMN39397539,3394419299,INSTITUT PASTEUR OF SHANGHAI,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6564,ERR15951191,WGS,212,10740246317,SAMEA120589881,3365993450,QUADRAM INSTITUTE BIOSCIENCE,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6565,ERR15951193,WGS,280,59183488003,SAMEA120589883,17798921408,QUADRAM INSTITUTE BIOSCIENCE,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6566,ERR15951194,WGS,271,35140445635,SAMEA120589884,10911327171,QUADRAM INSTITUTE BIOSCIENCE,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6567,ERR15951203,WGS,274,6685169405,SAMEA120589893,2044166040,QUADRAM INSTITUTE BIOSCIENCE,public,"fastq,run.zq,sra","gs,ncbi,s3",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Убираем пустые столбцы. В итоге остается 1189 архивов

In [ ]:
df = df.dropna(axis=1, how='all')
df.to_csv('../sra_filtered_no_16S.csv', index=False)

In [20]:
df['BioProject'].value_counts().to_csv('unique_projects.csv')

In [146]:
#df['tissue'].value_counts()
#df['HOST'].value_counts()
#df['Organism'].value_counts()
#df.loc[df['BioProject'] == 'PRJNA665586']
#df.loc[df['HOST'] == 'Environment']
#df.loc[df['isolation_source'] == 'forest']

### Данные весят 2.58 терабайт

In [20]:
df = pd.read_csv('../sra_filtered_no_16S.csv')
total_bytes = df['Bytes'].sum()
print(f"Общий объем данных: {total_bytes/1e12:.4f} TB")

Общий объем данных: 2.8610 TB


In [21]:
print("\nБольшие файлы:")
top_large = df.nlargest(10, 'Bytes')[['Run', 'Bytes', 'Assay Type', 'Organism']].copy()
top_large['Size_GB'] = top_large['Bytes'] / 1e9
top_large['Size_TB'] = top_large['Bytes'] / 1e12
print(top_large[['Run', 'Size_GB', 'Size_TB', 'Assay Type', 'Organism']])


Большие файлы:
              Run     Size_GB   Size_TB Assay Type              Organism
919   SRR23314745  161.519252  0.161519    RNA-Seq        bat metagenome
920   SRR23314746  123.813281  0.123813    RNA-Seq        bat metagenome
917   SRR23314743  100.071383  0.100071    RNA-Seq        bat metagenome
1150  ERR15951201   67.632926  0.067633        WGS    bat gut metagenome
921   SRR23314747   61.614715  0.061615    RNA-Seq        bat metagenome
925   SRR23314751   57.810831  0.057811    RNA-Seq        bat metagenome
924   SRR23314750   53.888778  0.053889    RNA-Seq        bat metagenome
923   SRR23314749   34.270534  0.034271    RNA-Seq        bat metagenome
495   SRR14860617   33.661649  0.033662        WGS        gut metagenome
842   SRR18679138   32.391888  0.032392    RNA-Seq  Alphacoronavirus sp.


In [18]:
categorical_cols = ['Assay Type', 'LibraryLayout', 'LibrarySource', 'Platform', 'geo_loc_name_country', 'geo_loc_name_country_continent', 'sample_type']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Количество уникальных значений: {df[col].nunique()}")
        print(f"  Топ-10 значений: {df[col].value_counts().head(10).to_dict()}")


Assay Type:
  Количество уникальных значений: 4
  Топ-10 значений: {'RNA-Seq': 4206, 'AMPLICON': 1235, 'WGS': 643, 'OTHER': 485}

LibraryLayout:
  Количество уникальных значений: 2
  Топ-10 значений: {'PAIRED': 5887, 'SINGLE': 682}

LibrarySource:
  Количество уникальных значений: 6
  Топ-10 значений: {'METATRANSCRIPTOMIC': 3900, 'METAGENOMIC': 1867, 'VIRAL RNA': 423, 'TRANSCRIPTOMIC': 177, 'GENOMIC': 142, 'OTHER': 60}

Platform:
  Количество уникальных значений: 7
  Топ-10 значений: {'DNBSEQ': 3708, 'ILLUMINA': 2690, 'OXFORD_NANOPORE': 156, 'BGISEQ': 9, 'CAPILLARY': 4, 'GENEMIND': 1, 'LS454': 1}

geo_loc_name_country:
  Количество уникальных значений: 39
  Топ-10 значений: {'China': 3752, 'Spain': 854, 'Kenya': 535, 'Uganda': 225, 'Brazil': 165, 'France': 129, 'USA': 129, 'uncalculated': 94, 'Cameroon': 88, 'Saudi Arabia': 72}

geo_loc_name_country_continent:
  Количество уникальных значений: 7
  Топ-10 значений: {'Asia': 3968, 'Europe': 1194, 'Africa': 889, 'South America': 165, 'No

In [4]:
df = pd.read_csv('../sra_filtered_no_16S.csv', index_col=False)

In [5]:
df

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,Bytes,Center Name,Consent,DATASTORE filetype,...,env_package,experimental_factor,feature,identified_by,label,material,sample_ID,Host_Diet,organism,rel_to_oxygen
0,SRR868695,WGS,168,32700916,PRJNA203532,SAMN02147246,24426068,UNIVERSITY OF TURKU,public,"fastq,run.zq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,sea coast,NaN,bat-fin-2018,feces,NaN,insectivorous,NaN,NaN
1,SRR975462,OTHER,200,1112399600,PRJNA218570,SAMN02351598,742222275,INSTITUTE OF MILITARY VETERINARY,public,"fastq,sra",...,MIGS/MIMS/MIMARKS.host-associated,NaN,feces,NaN,NaN,feces,NaN,NaN,NaN,NaN
2,SRR6202122,WGS,202,18066072,PRJNA415003,SAMN07811889,9331475,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SRR6202123,WGS,202,5645294,PRJNA415003,SAMN07811890,3525641,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRR6202124,WGS,202,6025458,PRJNA415003,SAMN07811891,3733079,BGI,public,"fastq,run.zq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1153,ERR15951205,WGS,258,37874548118,PRJEB103765,SAMEA120589895,11473741804,QUADRAM INSTITUTE BIOSCIENCE,public,"sra,run.zq,fastq",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1154,ERR15951206,WGS,275,57310186414,PRJEB103765,SAMEA120589896,17777169612,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1155,ERR15951207,WGS,279,67460067788,PRJEB103765,SAMEA120589897,20797734971,QUADRAM INSTITUTE BIOSCIENCE,public,"run.zq,fastq,sra",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1156,SRR37069663,WGS,7864,1978643193,PRJNA1417558,SAMN54999376,622989701,JILIN AGRICULTURAL UNIVERSITY,public,"bam,sra",...,NaN,NaN,NaN,Jilin Provincial Key Laboratory of Animal Reso...,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
import pandas as pd

accessions_file = "../data/60_analyzed_sras.txt"


# read accession list
with open(accessions_file) as f:
    accessions = [line.strip() for line in f if line.strip()]

# filter rows
filtered = df[df["Run"].isin(accessions)]

# select columns of interest
result = filtered[["Run", "BioProject", "geo_loc_name_country", "HOST"]]

# save or print
print(result)
result.to_csv("matched_sra_metadata.csv", index=False)

             Run    BioProject geo_loc_name_country  \
7     SRR6202127   PRJNA415003       United Kingdom   
13   SRR22460267   PRJNA906729               Russia   
29   SRR25884626   PRJNA997461                China   
41   SRR25884681   PRJNA997461                China   
49   SRR25884743   PRJNA997461                China   
71   SRR25884871   PRJNA997461                China   
98   SRR31341449  PRJNA1185808                China   
100  SRR31341451  PRJNA1185808                China   
121  SRR33227722  PRJNA1253095               Russia   
138  SRR33693691  PRJNA1267097             Cambodia   
166  SRR31363914  PRJNA1185806              Nigeria   
167  SRR31363915  PRJNA1185806              Nigeria   
193  SRR31861041  PRJNA1204523              Finland   
194  SRR31861042  PRJNA1204523              Finland   
259  SRR14381427   PRJNA707649                China   
261  SRR14381429   PRJNA707649                China   
274  SRR25921359  PRJNA1013252          South Korea   
290  SRR28